In [ ]:
# @markdown Run this cell to check this notebook's version.

current_version = "0.4.0"
notebook_name = "OpticalFlow3D_Profiler"

# First of all check if ipywidgets is installed, if not throw an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    
    try:
        import yaml
    except ImportError:
        raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

    import requests
    import base64

    # Create widgets
    github_repo = widgets.Text(
        value="",
        description="*GitHub Repository URL:",
        placeholder="e.g https://github.com/username/repository",
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    github_token = widgets.Password(
        value="",
        description="*Personal Access Token:",
        placeholder="[Disabled] e.g. ghp_XXXXXXXXXXXXXXXXXXXX",
        disabled=True,
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    test_button = widgets.Button(
        description="Test Connection",
        button_style='success',
    )

    output_area = widgets.HTML()

    # By default the repository is considered public
    if "repository_is_private" not in globals():
        repository_is_private = False
    elif repository_is_private:
        github_token.disabled = False
        github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"

    def on_test_button_clicked(b):
        global repository_is_private
        
        # Check that the GitHub repository URL is provided
        if not github_repo.value:
            output_area.value = "⚠️ Please provide a GitHub repository URL."
            return
        
        # Get the owner and repo name from the URL
        repo_url = github_repo.value.rstrip('/')
        github_owner, github_repo_name = repo_url.split('/')[-2:]
        
        # Online version checking file path
        version_file_path = "notebooks/notebook_latest_versions.yaml"
        version_url = f"https://api.github.com/repos/{github_owner}/{github_repo_name}/contents/{version_file_path}"

        # Do an initial request to check if the repository is public
        if not repository_is_private:
            version_response = requests.get(version_url)
            if version_response.status_code == 404:
                repository_is_private = True
                github_token.disabled = False
                github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"
                output_area.value = f"⚠️ We have detected that the repository might be private. Please provide a Personal Access Token and click 'Test Connection' again."
                return
        else:
            headers = {"Accept": "application/vnd.github.v3+json"}
            if not github_token.value:
                output_area.value = "⚠️ Personal Access Token is required for private repositories."
                return
            headers["Authorization"] = f"token {github_token.value}"

            version_response = requests.get(version_url, headers=headers)

        # Check the response status
        if version_response.status_code == 200:
            content = version_response.json()['content']
            decoded_content = base64.b64decode(content).decode('utf-8')
            config = yaml.safe_load(decoded_content)
            # notebook_latest_versions.yaml groups the notebooks by folder: {"<folder>": {"<notebook>": "<version>"}}
            latest_version = next((str(group[notebook_name]) for group in (config or {}).values()
                                   if isinstance(group, dict) and notebook_name in group), "")

            output_area.value = (f"<b>Notebook version:</b> `{current_version}`<br>"
                                f"<b>Latest version available:</b> `{latest_version}`<br>")

            if latest_version == "":
                output_area.value += "⚠️ This notebook is not listed in the version file.<br>"
            elif current_version == latest_version:
                output_area.value += "✅ This notebook is up-to-date.<br>"
            else:
                output_area.value += f"⚠️ A new version of this notebook is available."
        else:
            output_area.value += "⚠️ Could not retrieve the version file.<br>"

    test_button.on_click(on_test_button_clicked)
    # Widget layout
    widget_box = widgets.VBox([github_repo, github_token, test_button, output_area])
    display(widget_box)
except ImportError:
    display(widgets.HTML('To check the notebook version, please go to the <a href="../Welcome.ipynb" target="_blank" style="color: #0066cc; text-decoration: underline;">Welcome notebook</a>.'))


# OpticalFlow3D Profiler

Quantify nuclear motion in 3D fluorescence time-lapses with the original
[OpticalFlow3D](https://github.com/aicjanelia/OpticalFlow3D) algorithm (Janelia, BSD-3-Clause), turn the
velocity fields into motion features per field of view, and compare conditions with
[MorphoCoverage](https://github.com/CellMigrationLab/MorphoCoverage).

Run the sections in order. Every choice is saved in the run folder (`run_manifest.json`), so a result can
always be traced back to its images and settings.

**Read before interpreting results**
- Optical flow measures the motion of *intensity patterns*, not tracked nuclei or single-cell trajectories.
- Speeds are *relative optical-flow speeds*: on synthetic nuclei the 3D algorithm underestimates XY speed
  (×0.5–0.85) and overestimates Z speed, depending on nuclear size (see `docs/validation.md`). Compare
  conditions imaged with the same settings.
- The field of view is the observation and the biological repeat the replicate; time points are never
  replicates.

## Section 0 — Initialise

**Install** sets up the notebook. In the installed app everything is already there; on Google Colab it
installs the package (a minute or two). **Colab users:** run the optional cell after it to reach your
images on Google Drive (they then appear under `/content/gdrive/MyDrive`).

In [ ]:
# @title Install
import sys

if "google.colab" in sys.modules:
    # Colab only: the installed app already contains the package and its dependencies.
    # MorphoCoverage and morphobricks-normalization are pinned to the commits the app was
    # tested with (requirements.yaml).
    !pip install -q "git+https://github.com/guijacquemet/LabConstrictor-OpticalFlow3D" "morphocoverage @ git+https://github.com/CellMigrationLab/MorphoCoverage@209b9be40a14aef5a5e244a8d031a4828f1e96ef" "tqdm==4.70.1" "morphobricks-normalization @ git+https://github.com/CellMigrationLab/morphobricks-normalization@26112a5b5019d3c1af91450aeabf623a05d6ba36"
    from google.colab import output
    output.enable_custom_widget_manager()
else:
    print("Local Jupyter: the packages come with the installed app.")

%matplotlib inline
from IPython.display import display
from opticalflow3d_profiler.notebook_analysis_ui import build

app = build()
print("Ready.")

In [ ]:
# @title Optional — mount Google Drive (Colab only)
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/gdrive")
else:
    print("Not on Colab: nothing to mount; use folders on this computer.")

## Section 1 — Load the data

- **Experiment folder**, in the MorphoBricks layout:
  `<experiment>/<condition>/<biological repeat>/<technical repeat>/<image>.tif`. Every level is optional (a
  single folder of images works) and a missing level is recorded as `single`. One single-channel TZYX
  TIFF / OME-TIFF per field of view; `position_id` is the image name.
- **Results folder**: where runs and reports are written (outside the experiment folder; never overwritten).

**Load the experiment** lists the fields of view per condition and repeat and opens a random image to read
its header (axes, pixel sizes, frame interval). Check the pre-filled values, then **Confirm**: every image
header is checked against them before anything is computed.

In [ ]:
# @title Section 1 — Load the data
display(app["load"].widget())

## Section 2 — Choose the settings

- **Optical flow**: *2D slice per slice* (recommended when the Z step is much larger than the pixel size),
  *3D*, or automatic. `xyzSig`, `tSig` and `wSig` are the original algorithm's defaults (with `tSig=1`, 7
  time points give one output, so a 15-frame movie gives 9). Brightness correction is off unless you choose it.
- **Features**: an intensity-threshold foreground (columns named `<foreground name>__<feature>`) or the whole
  image (`whole__<feature>`), the reliability filter and the speed.

**Estimate resources** before a large experiment, then **Process the example**: the example image is
processed into `<results>/examples/` and its optical flow and features are shown, so you can adjust the
settings before processing everything.

In [ ]:
# @title Section 2 — Choose the settings
display(app["settings"].widget())

## Section 3 — Process the experiment

Optical flow and features for every field of view, with the settings of section 2, into a new run folder.
Failed fields of view are listed, never skipped. Two tables are written: `features_per_frame.csv` (every time
point `t`) and `features_per_fov.csv` (each feature averaged over time per field of view).

In [ ]:
# @title Section 3 — Process the experiment
display(app["process"].widget())

## Section 4 — Analyse the results

**Load the results** of section 3, or of earlier runs (one folder per line) to reload previous results.

In [ ]:
# @title Section 4a — Load the results
display(app["results"].widget())

**Experimental design.** Choose the **control** every other condition is compared with and the **readout**
(every feature is tested alike; the readout leads the report). Confirm that a biological-repeat label is the
same biological repeat in every condition: MorphoCoverage pairs repeats only on that confirmation.

In [ ]:
# @title Section 4b — Experimental design
display(app["design"].widget())

**Report.** MorphoCoverage compares each condition with the control, the field of view as the observation:
Cohen's d, the exploratory FOV-level randomization screen and the paired test on the repeat means, with the
interactive PCA/UMAP and each feature's figure. The link opens the HTML report in a new tab.

In [ ]:
# @title Section 4c — HTML report
display(app["report"].widget())

## Optional — try the app on synthetic data

Runs every section on a synthetic experiment with a known answer (`ctrl` nuclei move slowly in random
directions, `drug` nuclei twice as fast in one direction), checks the results and keeps every file, with
`demo_report.md` listing each expectation as PASS or FAIL. Use it to confirm a new installation works.

In [ ]:
# @title Optional — synthetic demo
display(app["demo"].widget())

---
**Please cite** the OpticalFlow3D paper when you use this app:

> Lee, R. M., Eisenman, L. R., Hobson, C. M., Aaron, J. S. and Chew, T.-L. (2026). OpticalFlow3D – a tool for measuring amorphous motion in three-dimensional fluorescence microscopy images. *J. Cell Sci.* 139(21), jcs264851. https://doi.org/10.1242/jcs.264851

and LabConstrictor: Hidalgo-Cenalmor, I. et al. (2026). Packaging Jupyter notebooks as installable desktop apps using LabConstrictor. arXiv. https://doi.org/10.48550/arXiv.2603.10704

The condition statistics and the report are MorphoCoverage's (CellMigrationLab, MIT licence).